# Limpieza

## Objetivo

Traducir programas escritos en el lenguaje "limpieza" a scripts Python que limpian texto copiado de páginas web.

## Alcance

**Para qué sirve:** el lenguaje *limpieza* permite pedir, con pocas instrucciones en español, cómo limpiar el texto de una página web (HTML). El traductor convierte el programa en un script de Python que lee el archivo, hace la limpieza y guarda el resultado en un `.txt`. Las acciones semánticas escriben el código Python de cada paso, y un agente inteligente (Ollama) interpreta las palabras aproximadas que escribe el usuario.

**Funcionalidades:**

* Carga un archivo `.txt` con HTML y guarda el resultado en otro `.txt`.
* `quitar etiquetas`: borra las etiquetas HTML y deja el texto (los cierres de párrafo, título pasan a salto de línea).
* `quitar espacios`: borra espacios repetidos, líneas vacías y espacios antes de la puntuación.
* `minusculas`: pasa el texto a minúsculas.
* `conservar clase X` / `conservar identificador X`: deja solo el texto de los elementos con esa clase o id.
* Entiende palabras aproximadas con Ollama (por ejemplo, `marcas` se interpreta como `etiquetas`).
* Detecta errores léxicos, sintácticos y semánticos.

**Reglas:**

* Todo programa empieza con `cargar`, tiene al menos un paso y termina con `guardar`.
* Cada paso hace solo lo que dice: sin `quitar etiquetas`, el HTML se conserva.
* `conservar` va antes de `quitar etiquetas` y se permite uno solo por programa.

**Fuera del alcance:**

* No descarga páginas: el HTML se copia a mano a un `.txt` (UTF-8).
* No quita URLs; un enlace `<a>` se trata como cualquier etiqueta (queda su texto, se pierde la dirección).
* `<br>` no genera salto de línea y no se decodifican entidades HTML.
* Ollama solo interpreta palabras: no genera el código de limpieza (eso lo escribe el programa).

## Especificaciones léxicas

| Token | Lexema | Patrón |
| --    | --     | --     |
| CARGAR | cargar | `cargar` |
| GUARDAR | guardar | `guardar` |
| QUITAR | quitar | `quitar` |
| MINUSCULAS | minusculas | `minusculas` |
| CONSERVAR | conservar | `conservar` |
| CLASE | clase | `clase` |
| IDENTIFICADOR | identificador | `identificador` |
| ARCHIVO | texto.txt | `[\w./-]+\.txt` |
| ID | etiquetas, novedades | `[a-zA-Z_][\w-]*` |

Las palabras reservadas no distinguen mayúsculas de minúsculas. Cualquier otra palabra que cumpla el patrón de ID es un ID.

## Especificaciones sintácticas

```text
<programa> ::= <carga> <pasos> <guardado>
<carga>    ::= CARGAR ARCHIVO
<pasos>    ::= <pasos> <paso>
<pasos>    ::= <paso>
<paso>     ::= QUITAR ID
<paso>     ::= MINUSCULAS
<paso>     ::= CONSERVAR CLASE ID
<paso>     ::= CONSERVAR IDENTIFICADOR ID
<guardado> ::= GUARDAR ARCHIVO
```

## Especificaciones semánticas

| Regla | Acción semántica |
|:------|:-----------------|
| `<carga> ::= CARGAR ARCHIVO` | Escribe la lectura del archivo. Error si el archivo no existe. |
| `<paso> ::= QUITAR ID` | Si ID no es `etiquetas` ni `espacios`, le pregunta a Ollama cuál se parece. Escribe el código Python de ese paso. Error si Ollama tampoco la entiende. |
| `<paso> ::= MINUSCULAS` | Escribe `texto = texto.lower()`. |
| `<paso> ::= CONSERVAR CLASE ID` | Escribe el código con BeautifulSoup que deja solo el texto de esa clase. Error si va después de `quitar etiquetas` o si ya hay otro `conservar`. |
| `<paso> ::= CONSERVAR IDENTIFICADOR ID` | Igual, pero con un id. |
| `<guardado> ::= GUARDAR ARCHIVO` | Escribe el guardado del resultado. Error si es el mismo archivo que se cargó. |

**Qué hace Ollama:** si el usuario escribe una palabra que no es válida, el parser le pregunta a Ollama a cuál opción se parece. Esto pasa en dos lugares:
* Después de `quitar`: si la palabra no es `etiquetas` ni `espacios` (por ejemplo `marcas` o `blancos`), Ollama decide a cuál se parece.
* Después de `conservar`: si la palabra no es `clase` ni `identificador` (por ejemplo `id`), Ollama decide a cuál se parece.

Si la respuesta no es una opción válida, se informa un error semántico. El código Python de cada paso lo escribe el parser, y leer y guardar archivos también.

## Implementación

In [2]:
%%writefile limpieza/lexer_limpieza.py
import ply.lex as lex

# palabras reservadas
reservadas = {
    'cargar': 'CARGAR',
    'guardar': 'GUARDAR',
    'quitar': 'QUITAR',
    'minusculas': 'MINUSCULAS',
    'conservar': 'CONSERVAR',
    'clase': 'CLASE',
    'identificador': 'IDENTIFICADOR',
}

tokens = ['ARCHIVO', 'ID'] + list(reservadas.values())

# errores léxicos
errores_lex = []


def t_ARCHIVO(t):
    r'[\w./-]+\.txt'
    return t


def t_ID(t):
    r'[a-zA-Z_][\w-]*'
    # si es reservada, cambia el token
    t.type = reservadas.get(t.value.lower(), 'ID')
    return t


# se ignoran espacios y tabs
t_ignore = ' \t'


def t_newline(t):
    r'\n+'
    t.lexer.lineno += len(t.value)


def t_error(t):
    errores_lex.append(
        f"Error léxico en línea {t.lineno}: carácter ilegal '{t.value[0]}'"
    )
    t.lexer.skip(1)


lexer = lex.lex()


Overwriting limpieza/lexer_limpieza.py


#### Agente (Ollama)

In [3]:
%%writefile limpieza/agente.py
import ollama

MODELO = 'qwen2.5-coder:1.5b'


def preguntar(pregunta):
    try:
        respuesta = ollama.chat(
            model=MODELO,
            options={'temperature': 0},
            messages=[{'role': 'user', 'content': pregunta}]
        )
    except Exception:
        return ''
    return respuesta['message']['content']


def entender_palabra(palabra, opciones):
    if palabra in opciones:
        return palabra

    respuesta = preguntar(
        f'Opciones válidas: {", ".join(opciones)}.\n'
        f'El usuario escribió "{palabra}".\n'
        'Respondé con UNA sola palabra: la opción que significa lo mismo, '
        'o NINGUNA si no se parece a ninguna.'
    ).lower()

    # opciones que aparecen en la respuesta
    encontradas = []
    for op in opciones:
        if op in respuesta:
            encontradas.append(op)

    # se acepta solo si hay una
    if len(encontradas) == 1:
        return encontradas[0]
    return None

Overwriting limpieza/agente.py


In [4]:
%%writefile limpieza/parser_limpieza.py
import os
import ply.yacc as yacc

import lexer_limpieza as L
from lexer_limpieza import tokens
from agente import entender_palabra

start = 'programa'

# codigo de cada paso de quitar 
CODIGOS = {
    'etiquetas': [
        "texto = re.sub(r'(?i)</(p|div|h[1-6]|li|tr)>', '\\n', texto)",
        "texto = re.sub(r'<[^>]*>', '', texto)",
    ],
    'espacios': [
        "texto = re.sub(r'[ \\t]+', ' ', texto)",
        "texto = re.sub(r'\\s*\\n\\s*', '\\n', texto)",
        "texto = re.sub(r'\\s+([.,;:!?])', r'\\1', texto)",
        "texto = texto.strip()",
    ],
}


estado = {
    'codigo': [],
    'errores': [],
    'archivo': None,
    'sin_etiquetas': False,
    'hay_filtro': False,
}


def agregar(nombre, lineas):
    estado['codigo'].append('# ' + nombre)
    estado['codigo'] += lineas


# reglas de la gramática

def p_programa(p):
    'programa : carga pasos guardado'


def p_carga(p):
    'carga : CARGAR ARCHIVO'
    estado['archivo'] = p[2]

    if not os.path.isfile(p[2]):
        estado['errores'].append(f"Error semántico: no existe el archivo '{p[2]}'")

    estado['codigo'] += [
        'import re',
        'from bs4 import BeautifulSoup',
        f'with open("{p[2]}", encoding="utf-8") as f:',
        '    texto = f.read()',
    ]


def p_pasos(p):
    '''pasos : pasos paso
             | paso'''


def p_paso_quitar(p):
    'paso : QUITAR ID'
    objetivo = entender_palabra(p[2].lower(), list(CODIGOS))

    if objetivo is None:
        estado['errores'].append(
            f"Error semántico: no se reconoce '{p[2]}'. "
            f"Opciones válidas: {', '.join(CODIGOS)}"
        )
        return

    if objetivo == 'etiquetas':
        estado['sin_etiquetas'] = True

    agregar('quitar ' + objetivo, CODIGOS[objetivo])


def p_paso_minusculas(p):
    'paso : MINUSCULAS'
    agregar('minusculas', ['texto = texto.lower()'])


def p_paso_conservar_clase(p):
    'paso : CONSERVAR CLASE ID'
    conservar('.' + p[3], f'class="{p[3]}"')


def p_paso_conservar_id(p):
    'paso : CONSERVAR IDENTIFICADOR ID'
    conservar('#' + p[3], f'id="{p[3]}"')
    
def p_paso_conservar_otra(p):
    'paso : CONSERVAR ID ID'
    # consulta a Ollama
    tipo = entender_palabra(p[2].lower(), ['clase', 'identificador'])

    if tipo is None:
        estado['errores'].append(
            f"Error semántico: no se reconoce '{p[2]}'. "
            "Opciones válidas: clase, identificador"
        )
    elif tipo == 'clase':
        conservar('.' + p[3], f'class="{p[3]}"')
    else:
        conservar('#' + p[3], f'id="{p[3]}"')

def p_guardado(p):
    'guardado : GUARDAR ARCHIVO'

    # no pisar el original
    if estado['archivo'] == p[2]:
        estado['errores'].append(
            "Error semántico: no se puede guardar sobre el archivo original"
        )

    estado['codigo'] += [
        f'with open("{p[2]}", "w", encoding="utf-8") as f:',
        '    f.write(texto)',
    ]


def p_error(p):
    if p is None:
        estado['errores'].append("Error sintáctico: el programa terminó antes de tiempo")
    else:
        estado['errores'].append(
            f"Error sintáctico en línea {p.lineno}: token inesperado '{p.value}'"
        )


def conservar(selector, atributo):
    if estado['sin_etiquetas']:
        estado['errores'].append(
            "Error semántico: 'conservar' tiene que ir antes de 'quitar etiquetas'"
        )
        return

    if estado['hay_filtro']:
        estado['errores'].append(
            "Error semántico: solo se permite un 'conservar' por programa"
        )
        return

    estado['hay_filtro'] = True

    agregar('conservar ' + selector, [
        "soup = BeautifulSoup(texto, 'html.parser')",
        f"texto = '\\n'.join(e.get_text(' ', strip=True) for e in soup.select('{selector}'))",
    ])


parser = yacc.yacc(debug=False, write_tables=False)


def compilar(texto):
    # reinicio
    estado['codigo'] = []
    estado['errores'] = []
    estado['archivo'] = None
    estado['sin_etiquetas'] = False
    estado['hay_filtro'] = False
    L.errores_lex.clear()
    L.lexer.lineno = 1

    parser.parse(texto, lexer=L.lexer)

    errores = L.errores_lex + estado['errores']

    if errores:
        return '', errores

    return '\n'.join(estado['codigo']) + '\n', []


Overwriting limpieza/parser_limpieza.py


#### Preparación: una función que traduce un programa, muestra el Python y lo ejecuta

In [5]:
import os, sys

RAIZ = os.getcwd()
sys.path.insert(0, os.path.join(RAIZ, 'limpieza'))

from parser_limpieza import compilar


def traducir_y_ejecutar(programa, salida, mostrar_python=False):
    os.chdir(os.path.join(RAIZ, 'limpieza', 'pruebas'))
    try:
        codigo, errores = compilar(programa)

        if errores:
            print('Se encontraron errores:')
            for error in errores:
                print(' ', error)
            return

        if mostrar_python:
            print('--- Programa traducido (Python) ---')
            print(codigo)

        exec(codigo, {})

        print('--- Resultado (' + salida + ') ---')
        print(open(salida, encoding='utf-8').read())
    finally:
        os.chdir(RAIZ)


## Ejemplo simple

In [6]:
%%writefile limpieza/pruebas/entrada.txt
<html>
<body>
<h1>MI TÍTULO</h1>
<p>Mira <a href="https://x.com">ESTE   sitio</a> .</p>
</body>
</html>


Overwriting limpieza/pruebas/entrada.txt


In [7]:
programa = """cargar entrada.txt
quitar espacios
guardar resultado.txt"""

traducir_y_ejecutar(programa, 'resultado.txt')
 

--- Resultado (resultado.txt) ---
<html>
<body>
<h1>MI TÍTULO</h1>
<p>Mira <a href="https://x.com">ESTE sitio</a>.</p>
</body>
</html>


Las líneas que empiezan con `#` en el programa traducido marcan cada paso de limpieza. El resto lo escribe el programa.

## Ejemplo complejo

La página tiene clases y un id. En el primer programa se escribe `conservar clase novedades`; en el segundo, la palabra aproximada `marcas`, que Ollama interpreta como `etiquetas`.

In [8]:
%%writefile limpieza/pruebas/pagina.txt
<html>
<body>
<h1>MI PÁGINA DE PRUEBA</h1>
<div class="novedades">Nuevo curso <a href="x">Mirá</a> el programa .</div>
<p>ESTE   ES UN TEXTO</p>
<p id="principal">Texto principal</p>
<div class="novedades">Segunda novedad</div>
</body>
</html>


Overwriting limpieza/pruebas/pagina.txt


In [9]:
programa = """cargar pagina.txt
conservar clase novedades
minusculas
guardar novedades.txt"""

traducir_y_ejecutar(programa, 'novedades.txt')


--- Resultado (novedades.txt) ---
nuevo curso mirá el programa .
segunda novedad


## Tests

Cada test se corre por separado: tiene su propio archivo de entrada, su programa *limpieza* y lo que se espera. Los tests 1 a 5 dan un resultado correcto; del 6 al 10 muestran los distintos tipos de error.

| Test | Qué prueba |
|:--|:--|
| 1 | El caso más simple |
| 2 | El caso complejo: varias clases y quedarse con un grupo |
| 3 | Qué pasa si no se pide `quitar etiquetas` |
| 4 | Qué pasa si solo se pide `minusculas` |
| 5 | Palabra aproximada (`marcas`), la interpreta Ollama |
| 6 | Error léxico: carácter ilegal |
| 7 | Error sintáctico: falta `guardar` |
| 8 | Error sintáctico: `sacar` en vez de `quitar` |
| 9 | Error semántico: el archivo no existe |
| 10 | Error semántico: `conservar` después de `quitar etiquetas` |

In [11]:
def crear(nombre, contenido):
    ruta = os.path.join(RAIZ, 'limpieza', 'pruebas', nombre)
    with open(ruta, 'w', encoding='utf-8') as f:
        f.write(contenido)


#### Test 1: el caso más simple

Una sola línea con una etiqueta. **Se espera:** `Hola mundo`.

In [12]:
crear('t1.txt', '<p>Hola mundo</p>')

programa = """cargar t1.txt
quitar etiquetas
guardar r1.txt"""

traducir_y_ejecutar(programa, 'r1.txt')


--- Resultado (r1.txt) ---
Hola mundo



#### Test 2: el caso complejo (varias clases, nos quedamos con un grupo)

La página tiene tres clases (`noticias`, `publicidad`, `menu`) y un id. Nos quedamos solo con la clase `noticias`, sin etiquetas, sin espacios de más y en minúsculas. **Se espera:** solo los textos de las dos noticias, en minúsculas.

In [13]:
crear('t2.txt', '''<html>
<body>
<div class="menu">INICIO   CONTACTO</div>
<div class="noticias">Nueva   BIBLIOTECA <b>abierta</b> .</div>
<div class="publicidad">COMPRÁ   YA</div>
<p id="pie">Todos los derechos</p>
<div class="noticias">Hoy hay   FERIA del libro</div>
</body>
</html>''')

programa = """cargar t2.txt
conservar clase noticias
quitar etiquetas
quitar espacios
minusculas
guardar r2.txt"""

traducir_y_ejecutar(programa, 'r2.txt')


--- Resultado (r2.txt) ---
nueva biblioteca abierta.
hoy hay feria del libro


#### Test 3: qué pasa si no se pide `quitar etiquetas`

Solo se piden `quitar espacios`. **Se espera:** las etiquetas siguen en el texto, porque cada paso hace solo lo que dice.

In [23]:
crear('t3.txt', """<html>
<head>
<title>Mi página</title>
</head>
<body>
<h1>Bienvenidos</h1>
<p>Hola    <b>mundo</b> .</p>
</body>
</html>""")

programa = """cargar t3.txt
quitar espacios
guardar r3.txt"""

traducir_y_ejecutar(programa, 'r3.txt')


--- Resultado (r3.txt) ---
<html>
<head>
<title>Mi página</title>
</head>
<body>
<h1>Bienvenidos</h1>
<p>Hola <b>mundo</b>.</p>
</body>
</html>


#### Test 4: qué pasa si solo se pide `minusculas`

**Se espera:** todo en minúsculas, pero con las etiquetas intactas.

In [15]:
crear('t4.txt', """<html>
<body>
<h1>MI TÍTULO</h1>
<p>Mira <a href="https://x.com">ESTE   sitio</a> .</p>
</body>
</html>""")

programa = """cargar t4.txt
minusculas
guardar r4.txt"""

traducir_y_ejecutar(programa, 'r4.txt')


--- Resultado (r4.txt) ---
<html>
<body>
<h1>mi título</h1>
<p>mira <a href="https://x.com">este   sitio</a> .</p>
</body>
</html>


#### Test 5: palabra aproximada, la interpreta Ollama

Se escribe `marcas` en vez de `etiquetas`. Ollama tiene que estar abierto. **Se espera:** que lo entienda como `etiquetas` y quite las etiquetas.

In [16]:
crear('t5.txt', """<html>
<head>
<title>Mi página</title>
</head>
<body>
<h1>Bienvenidos</h1>
<p>Hola<b>mundo</b> .</p>
</body>
</html>""")

programa = """cargar t5.txt
quitar marcas
guardar r5.txt"""

traducir_y_ejecutar(programa, 'r5.txt')


--- Resultado (r5.txt) ---


Mi página


Bienvenidos

Holamundo .





#### Test 6: error léxico (carácter ilegal)

El programa tiene un id , que no pertenece al lenguaje. **Se espera:** error léxico.

In [17]:
crear('t6.txt', '<p>Hola</p>')

programa = """cargar t6.txt
quitar etiquetas <p>
guardar r6.txt"""

traducir_y_ejecutar(programa, 'r6.txt')


Se encontraron errores:
  Error léxico en línea 2: carácter ilegal '<'
  Error léxico en línea 2: carácter ilegal '>'
  Error sintáctico en línea 2: token inesperado 'p'


#### Test 7: error sintáctico (falta `guardar`)

El programa termina sin la línea `guardar`. **Se espera:** error sintáctico.

In [18]:
crear('t7.txt', '<p>Hola</p>')

programa = """cargar t7.txt
quitar etiquetas"""

traducir_y_ejecutar(programa, 'r7.txt')


Se encontraron errores:
  Error sintáctico: el programa terminó antes de tiempo


#### Test 8: error sintáctico (`sacar` en vez de `quitar`)

La palabra reservada es `quitar`. El lexer lee `sacar` como un nombre cualquiera y el parser no la espera en ese lugar. **Se espera:** error sintáctico.

In [19]:
crear('t8.txt', '<p>Hola</p>')

programa = """cargar t8.txt
sacar etiquetas
guardar r8.txt"""

traducir_y_ejecutar(programa, 'r8.txt')


Se encontraron errores:
  Error sintáctico en línea 2: token inesperado 'sacar'


#### Test 9: error semántico (archivo que no existe)

La gramática está bien, pero `no_existe.txt` no está en la carpeta. **Se espera:** error semántico.

In [20]:
programa = """cargar no_existe.txt
quitar etiquetas
guardar r9.txt"""

traducir_y_ejecutar(programa, 'r9.txt')


Se encontraron errores:
  Error semántico: no existe el archivo 'no_existe.txt'


#### Test 10: error semántico (`conservar` después de `quitar etiquetas`)

Después de quitar las etiquetas ya no hay clases para buscar. **Se espera:** error semántico.

In [21]:
crear('t10.txt', '<div class="a">Hola</div>')

programa = """cargar t10.txt
quitar etiquetas
conservar clase a
guardar r10.txt"""

traducir_y_ejecutar(programa, 'r10.txt')


Se encontraron errores:
  Error semántico: 'conservar' tiene que ir antes de 'quitar etiquetas'


In [22]:
crear('t11.txt', """<html>
<body>
<h1>Mi página</h1>
<div id="menu">INICIO   CONTACTO</div>
<p id="principal">Este es el   TEXTO principal .</p>
<div id="pie">Todos los derechos</div>
</body>
</html>""")

programa = """cargar t11.txt
conservar id principal
quitar espacios
minusculas
guardar r11.txt"""

traducir_y_ejecutar(programa, 'r11.txt')

--- Resultado (r11.txt) ---
este es el texto principal.
